# Bước 1. Import thư viện và đọc lại Train/Test

In [1]:

import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "raw"

train_df = pd.read_csv(DATA_DIR / "youtube_train.csv")
test_df = pd.read_csv(DATA_DIR / "youtube_test.csv")

LEAKAGE_COLS = [
    "view_count",
    "like_count",
    "comment_count",
    "crawled_at",
    "channel_median_views",
    "viral_ratio",
    "viral_score_log",
    "engagement_rate"
]

y_train = train_df["is_viral"].astype(int)
y_test = test_df["is_viral"].astype(int)

X_train = train_df.drop(
    columns=LEAKAGE_COLS + ["is_viral"]
)

X_test = test_df.drop(
    columns=LEAKAGE_COLS + ["is_viral"]
)

assert X_train.columns.equals(X_test.columns)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)


X_train: (31058, 32)
X_test: (7765, 32)


# Bước 2. Lựa chọn các cột thực sự được dùng


# Bước 3. Chia Train thành Fit và Validation

In [2]:

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

fit_idx, val_idx = next(
    splitter.split(
        X_train,
        y_train,
        groups=train_df["channel_id"]
    )
)

X_fit_raw = X_train.iloc[fit_idx].copy()
X_val_raw = X_train.iloc[val_idx].copy()

y_fit = y_train.iloc[fit_idx].copy()
y_val = y_train.iloc[val_idx].copy()

print("Fit:", X_fit_raw.shape)
print("Validation:", X_val_raw.shape)

print("Fit viral ratio:", round(y_fit.mean(), 4))
print("Validation viral ratio:", round(y_val.mean(), 4))

# Kiểm tra các kênh có trùng nhau không
fit_channels = set(
    train_df.iloc[fit_idx]["channel_id"]
)

val_channels = set(
    train_df.iloc[val_idx]["channel_id"]
)

assert len(fit_channels & val_channels) == 0

print("Không có kênh trùng giữa Fit/Validation")


Fit: (24687, 32)
Validation: (6371, 32)
Fit viral ratio: 0.1818
Validation viral ratio: 0.1906
Không có kênh trùng giữa Fit/Validation


# Bước 4. Xây dựng hàm xử lý đặc trưng cơ bản

In [3]:

NUM_COLS = [
    "duration_sec",
    "tag_count",
    "title_char_len",
    "desc_visible_len",
    "has_description",
    "has_tags_text",
    "title_has_number",
    "publish_hour",
    "publish_dow",
    "channel_age_days"
]

CAT_COLS = [
    "category_name",
    "channel_country",
    "definition",
    "caption",
    "licensed_content",
    "made_for_kids",
    "default_language",
    "default_audio_language"
]

def make_basic_features(df):
    z = pd.DataFrame(index=df.index)

    # --- A. Xử lý Text ---
    title = df["title"].fillna("").astype(str)
    description = df["description"].fillna("").astype(str)
    tags = df["tags"].fillna("").astype(str)

    # --- B. Numerical features ---
    z["duration_sec"] = pd.to_numeric(
        df["duration_sec"], errors="coerce"
    )

    z["tag_count"] = pd.to_numeric(
        df["tag_count"], errors="coerce"
    )

    # --- C. Text-based features ---
    z["title_char_len"] = title.str.len()

    z["desc_visible_len"] = description.str.len()

    z["has_description"] = (
        description.str.strip().ne("")
    ).astype(int)

    z["has_tags_text"] = (
        tags.str.strip().ne("")
    ).astype(int)

    z["title_has_number"] = (
        title.str.contains(r"\d", regex=True)
    ).astype(int)

    # --- D. Datetime features ---
    published = pd.to_datetime(
        df["published_at"],
        utc=True,
        errors="coerce",
        format="mixed"
    )

    channel_created = pd.to_datetime(
        df["channel_published_at"],
        utc=True,
        errors="coerce",
        format="mixed"
    )

    # Đổi UTC sang giờ Việt Nam
    local_time = published.dt.tz_convert(
        "Asia/Ho_Chi_Minh"
    )

    z["publish_hour"] = local_time.dt.hour
    z["publish_dow"] = local_time.dt.dayofweek

    # Tuổi kênh tại thời điểm video được đăng
    z["channel_age_days"] = (
        (published - channel_created)
        .dt.total_seconds()
        .div(86400)
        .clip(lower=0)
    )

    # --- E. Categorical features ---
    for col in CAT_COLS:
        z[col] = (
            df[col]
            .astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )

    return z


# Bước 5. Áp dụng hàm tạo đặc trưng

In [4]:

F_fit = make_basic_features(X_fit_raw)
F_val = make_basic_features(X_val_raw)

print("Fit features:", F_fit.shape)
print("Validation features:", F_val.shape)

print("\nCác cột được tạo:")
print(F_fit.columns.tolist())

display(F_fit.head())

# Kiểm tra các cột có thống nhất không
assert F_fit.columns.equals(F_val.columns)

# Kiểm tra feature số và categorical
assert set(F_fit.columns) == set(NUM_COLS + CAT_COLS)


Fit features: (24687, 18)
Validation features: (6371, 18)

Các cột được tạo:
['duration_sec', 'tag_count', 'title_char_len', 'desc_visible_len', 'has_description', 'has_tags_text', 'title_has_number', 'publish_hour', 'publish_dow', 'channel_age_days', 'category_name', 'channel_country', 'definition', 'caption', 'licensed_content', 'made_for_kids', 'default_language', 'default_audio_language']


C:\Users\Administrator\AppData\Local\Temp\ipykernel_18724\342131061.py:80: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  z["publish_dow"] = local_time.dt.dayofweek
C:\Users\Administrator\AppData\Local\Temp\ipykernel_18724\342131061.py:80: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  z["publish_dow"] = local_time.dt.dayofweek


,duration_sec,tag_count,title_char_len,desc_visible_len,has_description,has_tags_text,title_has_number,publish_hour,publish_dow,channel_age_days,category_name,channel_country,definition,caption,licensed_content,made_for_kids,default_language,default_audio_language
1,1402,5,53,500,1,1,0,20,2,1859.138366,Education,VN,hd,False,True,False,vi,vi
2,161,24,53,0,0,1,1,21,5,4451.092928,Science & Technology,VN,hd,False,False,False,vi,vi
3,355,13,56,376,1,1,1,11,5,2752.691736,Science & Technology,VN,hd,False,True,False,vi,zxx
4,1682,24,73,492,1,1,0,19,2,340.294230,Film & Animation,VN,hd,False,True,False,vi,vi
5,1057,13,9,500,1,1,0,19,1,4554.785718,Howto & Style,VN,hd,False,True,False,vi,vi


# Bước 6. Thiết kế Preprocessing Pipeline
# 6.1. Xử lý Missing Values
# 6.2. One-Hot Encoding
# 6.3. Khởi tạo ColumnTransformer

In [5]:

numeric_transformer = SimpleImputer(
    strategy="median"
)

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            NUM_COLS
        ),
        (
            "cat",
            categorical_transformer,
            CAT_COLS
        )
    ],
    remainder="drop"
)

print(preprocessor)


ColumnTransformer(transformers=[('num', SimpleImputer(strategy='median'),
                                 ['duration_sec', 'tag_count', 'title_char_len',
                                  'desc_visible_len', 'has_description',
                                  'has_tags_text', 'title_has_number',
                                  'publish_hour', 'publish_dow',
                                  'channel_age_days']),
                                ('cat',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['category_name', 'channel_country',
                                  'definition', 'caption', 'licensed_content',
                                  'made_for_kids', 'default_language',
                                  'default_audio_language'])])


# Bước 7. Fit Pipeline và biến đổi dữ liệu

In [6]:

# Học preprocessing từ Fit
X_fit_ready = preprocessor.fit_transform(F_fit)

# Chỉ áp dụng phép biến đổi đã học lên Validation
X_val_ready = preprocessor.transform(F_val)

print("X_fit_ready:", X_fit_ready.shape)
print("X_val_ready:", X_val_ready.shape)

print("y_fit:", y_fit.shape)
print("y_val:", y_val.shape)


X_fit_ready: (24687, 48)
X_val_ready: (6371, 48)
y_fit: (24687,)
y_val: (6371,)


# Bước 8. Kiểm tra kết quả Preprocessing

In [7]:

# Kiểm tra NaN và giá trị vô hạn
assert np.isfinite(X_fit_ready).all()
assert np.isfinite(X_val_ready).all()

# Kiểm tra số feature khớp nhau
assert X_fit_ready.shape[1] == X_val_ready.shape[1]

# Kiểm tra số dòng với nhãn
assert len(X_fit_ready) == len(y_fit)
assert len(X_val_ready) == len(y_val)

print("Preprocessing thành công!")

# Lấy tên các đặc trưng sau mã hóa
feature_names = preprocessor.get_feature_names_out()

print("\nSố feature sau encoding:", len(feature_names))

# Hiển thị vài dòng kết quả
processed_preview = pd.DataFrame(
    X_fit_ready[:5],
    columns=feature_names
)

display(processed_preview)


Preprocessing thành công!

Số feature sau encoding: 48


,num__duration_sec,num__tag_count,num__title_char_len,num__desc_visible_len,num__has_description,num__has_tags_text,num__title_has_number,num__publish_hour,num__publish_dow,num__channel_age_days,...,cat__default_audio_language_ase,cat__default_audio_language_en,cat__default_audio_language_en-GB,cat__default_audio_language_en-IE,cat__default_audio_language_en-US,cat__default_audio_language_hi,cat__default_audio_language_id,cat__default_audio_language_ko,cat__default_audio_language_vi,cat__default_audio_language_zxx
0,1402.0,5.0,53.0,500.0,1.0,1.0,0.0,20.0,2.0,1859.138366,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,161.0,24.0,53.0,0.0,0.0,1.0,1.0,21.0,5.0,4451.092928,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,355.0,13.0,56.0,376.0,1.0,1.0,1.0,11.0,5.0,2752.691736,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1682.0,24.0,73.0,492.0,1.0,1.0,0.0,19.0,2.0,340.294230,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,1057.0,13.0,9.0,500.0,1.0,1.0,0.0,19.0,1.0,4554.785718,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
